In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:22px;}

</style>
"""))

# 1. knowlege base 구성을 위한 데이터 생성

In [2]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("./tex_docx/소득세법_with_markdown.docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 문서를 쪼갤때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n","\n"," ",""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n","\n"," ",""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

<timed exec>:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.


CPU times: total: 4.06 s
Wall time: 4.4 s


180

In [3]:
documents[43]

Document(metadata={'source': './tex_docx/소득세법_with_markdown.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n| 종합소득 과세표준 | 세율 |\n\n|---|---|\n\n| 1,400만원 이하 | 과세표준의 6퍼센트 |\n\n| 1,400만원 초과 5,000만원 이하 | 84만원 + (1,400만원을 초과하는 금액의 15퍼센트) |\n\n| 5,000만원 초과 8,800만원 이하 | 624만원 + (5,000만원을 초과하는 금액의 24퍼센트) |\n\n| 8,800만원 초과 1억5천만원 이하 | 1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트) |\n\n| 1억5천만원 초과 3억원

In [1]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv(override=True)
embedding = UpstageEmbeddings(
                model = 'embedding-query'
)

In [2]:
len(embedding.embed_query("소득세법"))

4096

In [3]:
from pinecone import Pinecone
pc = Pinecone()
[index.name for index in pc.list_indexes()]

['tax-index-markdown', 'tax-index-table', 'tax-index-openai', 'tax-index']

In [4]:
# pinecone의 index 생성하기
import time
from pinecone import Pinecone,ServerlessSpec
#pinecone 객체 생성
pc = Pinecone()# api_key=os.getenv('PINECONE_API_KEY')

# 1. 인덱스 설정
index_name = 'tax-index-markdown'
dimension = len(embedding.embed_query("차원확인용"))

# 2. 인덱스 생성(해당인덱스가 없을경우)
existing_indexes=[index.name for index in pc.list_indexes()]
if index_name not in existing_indexes:
    pc.create_index(
        name = index_name,
        dimension=dimension,
        metric='cosine', # 유사도 측정방식
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )
    # 인덱스가 준비될때까지 대기
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1) # 1초대기
    print(f'인덱스{index_name}를 생성하였습니다.')
else:
    print(f'인덱스{index_name}이 이미 존재합니다.')

인덱스tax-index-markdown이 이미 존재합니다.


In [5]:
index_name = 'tax-index-table'
pc.describe_index(index_name).status

{'ready': True, 'state': 'Ready'}

In [6]:
# 3. 인덱스 정보확인
print([index_name for index in pc.list_indexes()])
print(pc.describe_index(index_name).status['ready'])
index = pc.Index(index_name)
print(index.describe_index_stats())

['tax-index-table', 'tax-index-table', 'tax-index-table', 'tax-index-table']
True
{'dimension': 4096,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 180}},
 'total_vector_count': 180,
 'vector_type': 'dense'}


In [7]:
%%time
# 4.문서를 임배딩해서 vector DB(index)에 저장
# 처음 한 번
from pinecone import Pinecone
pc=Pinecone()
from langchain_pinecone import PineconeVectorStore
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )
# 업로드한 벡터DB를 load할때 
index_name = 'tax-index-table'
database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

CPU times: total: 93.8 ms
Wall time: 152 ms


# 2. 답변 생성 전 retrieval 확인

In [8]:
query = "연봉이 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query,k=2)# 기본값은 4

In [9]:
len(retrieved_docs)

2

In [10]:
retrieved_docs[0]

Document(id='05648888-c777-42a4-9c8d-67f300a1fcaa', metadata={'source': './tex_docx/소득세법_with_table.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1400만원 이하\n\n과세표준의 6퍼센트\n\n1400만원초과 5000만원 이하\n\n84만원 + (1,400만원을 초과하는 금액의 15퍼센트)\n\n5000만원초과 8800만원 이하\n\n624만원 + (5,000만원을 초과하는 금액의 24퍼센트)\n\n8800만원초과 1억5000만원 이하\n\n1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)\n\n1억5000만원초과

In [11]:
retriever = database.as_retriever(
    search_type='similarity',
    search_kwargs={'k':2}
)
retrieved_docs = retriever.invoke("연봉이 5천만원 직장인의 소득세는 얼마인가요?")
retrieved_docs[0]

Document(id='05648888-c777-42a4-9c8d-67f300a1fcaa', metadata={'source': './tex_docx/소득세법_with_table.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1400만원 이하\n\n과세표준의 6퍼센트\n\n1400만원초과 5000만원 이하\n\n84만원 + (1,400만원을 초과하는 금액의 15퍼센트)\n\n5000만원초과 8800만원 이하\n\n624만원 + (5,000만원을 초과하는 금액의 24퍼센트)\n\n8800만원초과 1억5000만원 이하\n\n1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)\n\n1억5000만원초과

In [12]:
retrieved_doc = "\n--\n".join([doc.page_content for doc in retrieved_docs])
#print(retrieved_docs)

In [13]:
'55조' in retrieved_doc

True

# 3.답변 생성 

In [14]:
from openai import OpenAI
client = OpenAI()
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
response = client.chat.completions.create(
    model = 'gpt-4o-mini',
    messages=[
        {'role':'system','content':'당신은 최고의 한국 소득세법 전문가입니다.'},
        {
            'role':'user',
            'content':f"""-[context]를 참고해서 사용자의 질문에 답변해 주세요.
            -[context]:{retrieved_doc}
            - 질문 : {query}"""
        }
    ],
    temperature=0.2
)

In [15]:
result = response.choices[0].message.content
print(result)

연봉 5천만원인 직장인의 소득세를 계산하기 위해서는 먼저 종합소득 과세표준에 따라 세율을 적용해야 합니다.

1. **과세표준 계산**: 연봉 5천만원은 종합소득 과세표준으로 사용됩니다.

2. **세율 적용**:
   - 5천만원은 1,400만원 초과 5,000만원 이하 구간에 해당합니다.
   - 이 구간의 세율은 다음과 같습니다:
     - 84만원 + (1,400만원을 초과하는 금액의 15%)

3. **계산**:
   - 5,000만원 - 1,400만원 = 3,600만원
   - 세액 = 84만원 + (3,600만원 × 15%)
   - 세액 = 84만원 + 540만원 = 624만원

따라서, 연봉 5천만원인 직장인의 소득세는 **624만원**입니다.


In [16]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model = 'gpt-4.1-mini')

In [17]:
from langchain_upstage import ChatUpstage
llm = ChatUpstage(
    model='solar-pro4',
    reasoning_effort='low' # 느리지만 더 깊게 추론하여 답변(low,medium,high)
) 

In [18]:
prompt = f"""[identity]
-당신은 최고의 한국 소득세법 전문가입니다.
-[context]를 참고하여 사용자의 질문에 답변해 주세요. [context]는 다음과 같아요
{retrieved_doc}
-질문:{query}"""

In [19]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

제공해주신 소득세법 조문(제55조 세율, 제59조 근로소득세액공제)을 바탕으로 설명해 드리겠습니다. 다만, **소득세는 총급여액(연봉)에서 각종 공제와 세액공제를 거쳐야 최종세액이 결정되므로**, 제공된 맥락만으로는 단일 정답을 산출할 수 없습니다. 아래에 법정 계산 구조와 제공된 조문을 활용한 예시 산정 방법을 정리해 드립니다.

### 1. 소득세 계산의 기본 흐름 (제공 조문 기준)
| 단계 | 내용 | 관련 조문 |
|------|------|-----------|
| ① 소득금액 산정 | 총급여 – 근로소득공제 – 기타 필요경비 | (제공 맥락에는 미포함, 소득세법 제47조 등) |
| ② 과세표준 산정 | 소득금액 – 인적공제·특별소득공제·신용카드공제 등 | (제공 맥락에는 미포함) |
| ③ 산출세액 계산 | 과세표준 × **제55조 세율** | 제55조 ① |
| ④ 세액공제 적용 | 산출세액 – 근로소득세액공제(제59조), 배당세액공제(제56조), 재해손실세액공제(제58조) 등 | 제59조, 제56조, 제58조 |
| ⑤ 결정세액 | ④ 결과값 (지방소득세 10%는 별도 고지) | – |

> ⚠️ **중요**: 제55조 세율은 **총급여가 아닌 `종합소득과세표준`**에 적용됩니다. 연봉 5,000만 원이라도 근로소득공제, 4대 보험료 공제, 기본공제, 카드·의료비·교육비 등 특별소득공제 등을 거치면 과세표준은 보통 3,000만 원 내외 수준으로 줄어듭니다.

---

### 2. 제공 조문만을 활용한 예시 계산
아래 계산은 **최종 과세표준이 5,000만 원으로 확정되었다는 가정** 하에 제55조와 제59조만 적용한 예시입니다. 실제 과세표준은 위와 같이 훨씬 낮아지는 경우가 일반적입니다.

#### ① 종합소득세 산출세액 (제55조 적용)
과세표준 5,000만 원은 `1,400만 원 초과 ~ 5,000만 원 이하` 구간에 해당합니다.
```
산출세액 = 84만 원 + (5,000만 원 - 1,400만 원) × 15%
         = 84

In [30]:
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_upstage import ChatUpstage,UpstageEmbeddings
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 1. 임배딩 & LLM 초기화
load_dotenv()
# embedding = OpenAIEmbeddings(model="text-embedding-3-large")
# llm = ChatOpenAI(model='gpt-4.1-mini')

embedding = UpstageEmbeddings(model='embedding-query')
llm = ChatUpstage(model='solar-pro4')

# 2. vector DB load
# pc=Pinecone()
index_name='tax-index-table'
vectorStore = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

# 3. Retriever 생성
# (1)search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(search_type='similarity',
                                    search_kwarge={'k':2}) # 유사도가 높은 top-3추출
# (2)search_type='mmr':
# retriever = vectorStore.as_retriever(search_type='mmr',
#                                     search_kwarge={'k':3,
#                                                   'fetch_k':10,
#                                                   'lambda_mult':0.5})# 1에 가까울수록 관련성 깊은 계산
# (3)search_type='similarity_score_threshold' :
# retriever = vectorStore.as_retriever(search_type='similarity_score_threshold',
#                                     search_kwarge={'k':3,
#                                                   'score_threshold':0.4})

# 4. 프롬프트템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다.다음 문맥을 참고하여 질문에 답하세요. 
답을 모르면 모른다고 답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 :"""
prompt = ChatPromptTemplate.from_template(template)

# 5. 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n',' ') for doc in retrieved_docs])



In [31]:
# 6. RAG 랭체인 구성(LCEL방식)
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    | prompt
    | llm
    | StrOutputParser()
)
result = rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')
print(result)

주어진 문맥만으로는 연봉 5천만 원에 대한 정확한 소득세액을 특정할 수 없습니다. 소득세액은 과세표준에서 근로소득공제, 인적공제, 연금보험료 공제 등을 차감한 후 세율을 적용하고, 다시 근로소득세액공제 등을 적용해 최종 결정되기 때문에 개인의 공제 항목에 따라 달라집니다. 따라서 연말정산 상황이나 공제 내역을 알 수 없는 상태에서 단순 연봉만으로 정확한 세액을 산출하는 것은 불가능합니다.


# 4. 키워드 사전 활용

In [32]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
# query = "종합소득이 5,000만원인 거주자의 소득세는 얼마인가요?"

In [33]:
import numpy as np
def cosine_similarity(vec1,vec2):
    '두 백터의 코사인 유사도를 반환'
    dot_product = np.dot(vec1,vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    if norm_vec1==0 or norm_vec2==0:
        return 0
    return dot_product / (norm_vec1*norm_vec2)

vec1 = embedding.embed_query('직장인')
vec2 = embedding.embed_query('거주자')
cosine_similarity(vec1,vec2)

np.float64(0.5402861885922909)

In [34]:
vec1 = embedding.embed_query('연봉')
vec2 = embedding.embed_query('종합소득')
cosine_similarity(vec1,vec2)

np.float64(0.566316344244509)

In [35]:
vec1 = embedding.embed_query('5천')
vec2 = embedding.embed_query('5,000')
cosine_similarity(vec1,vec2)

np.float64(0.8728291248383045)

In [39]:
# 키워드사전 : 일반사용자 단어 -> 소득세법용어
from langchain_core.prompts import ChatPromptTemplate
dictionary = ["연봉->종합소득","사람을 나타내는 표현->거주자","5천->5,000"]
prompt = ChatPromptTemplate.from_template(f"""사용자의 질문을 보고,우리의 사전을 참고해서 사용자 질문을 
변경해 주세요. 만약 변경할 필요가 없을 경우, 사용자의 질문을 변경하지않아도 됩니다.오직 질문만 리턴해주세요
-사전 : {dictionary}
-질문 : {{question}}""")

In [40]:
strOutputParser = StrOutputParser()
strOutputParser.invoke(llm.invoke(prompt.invoke("소득이 높은 남자가 있습니다.")))

'종합소득이 높은 거주자가 있습니다.'

In [41]:
# 위를 langchain으로
dictionary_chain = prompt | llm | StrOutputParser()
dictionary_chain.invoke({"question":"연봉이 5천만원인 직장인은?"})

'종합소득이 5,000만원인 거주자는?'

In [42]:
new_chain = dictionary_chain | rag_chain

In [43]:
new_chain.invoke("연봉이 5천만원인 직장인의 소득세는 얼마에요?")

'제시된 문맥에는 종합소득에서 소득공제를 차감한 종합소득과세표준 계산 규정과 세율표만 있을 뿐, 구체적인 공제 내역이 없으므로 정확한 소득세를 산출할 수 없습니다. 다만 종합소득과세표준이 5,000만원이라면 해당 세율표에 따라 84만원 + (5,000만원 - 1,400만원) × 15% = 1,476만원의 종합소득산출세액이 계산됩니다. 종합소득과세표준이 5,000만원이라는 가정하에는 약 1,476만원이며, 실제 세액은 각종 공제 및 세액공제를 반영한 과세표준에 따라 달라집니다.'

In [44]:
# 입력 {"question":"연봉이 5천만원인 직장인의 소득세는 얼마예요?"}
#       ↓
# dictionary_chain에 의해 질문 수정(documents용어 수정)
#       ↓
#  "개선된 질문"
#       ↓
#  retriever
#       ↓ 
# format_documents
#       ↓ 
# rag_chain의 prompt + llm
#       ↓ 
# strOutputParser()
 